In [ ]:
# ── Colab Setup (run this first) ──
!pip install openai pandas requests tabulate -q

import os
# Option 1: Paste key directly
os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

Notebook 01 — Data Ingestion & Exploration
AutoDataEngineer: Pull Real Messy Data + See What's Wrong

WHAT THIS DOES:
- Pulls real product data from Open Food Facts API (free, no key needed)
- Falls back to synthetic messy data if API is unavailable
- Shows exactly how messy the data is (the "before" picture)

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

### Setup

In [ ]:
import os
import json
import random
import requests
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print("Setup complete")

### Pull data from Open Food Facts API

In [ ]:
# This is REAL messy data — inconsistent units, missing fields, mixed formats.
# No API key needed. Free and open source.

def fetch_openfoodfacts(num_pages=5, page_size=50):
    """Fetch product data from Open Food Facts API."""
    all_products = []
    base_url = "https://world.openfoodfacts.org/cgi/search.pl"

    for page in range(1, num_pages + 1):
        print(f"  Fetching page {page}/{num_pages}...")
        try:
            resp = requests.get(base_url, params={
                "action": "process",
                "json": "true",
                "page": page,
                "page_size": page_size,
                "fields": "product_name,brands,categories,quantity,serving_size,"
                          "energy_100g,fat_100g,sugars_100g,proteins_100g,"
                          "nutrition_grade_fr,countries,last_modified_datetime,"
                          "code,packaging",
            }, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            products = data.get("products", [])
            all_products.extend(products)
        except Exception as e:
            print(f"  Page {page} failed: {e}")
            continue

    return all_products

print("Pulling data from Open Food Facts API...")
products = fetch_openfoodfacts(num_pages=5, page_size=50)

if len(products) >= 50:
    df_raw = pd.DataFrame(products)
    print(f"Got {len(df_raw)} products from API")
    DATA_SOURCE = "Open Food Facts API"
else:
    print("API unavailable or insufficient data. Using synthetic fallback.")
    DATA_SOURCE = "Synthetic (API fallback)"
    df_raw = None

### Fallback — Generate synthetic messy data

In [ ]:
# If the API didn't return enough data, create realistic messy data.
# This data mimics real-world mess: mixed formats, nulls, inconsistencies.

def generate_messy_data(n=200):
    """Generate intentionally messy product data."""
    random.seed(42)
    np.random.seed(42)

    names = ["Organic Oat Milk", "Chocolate Bar", "Greek Yogurt", "Whole Wheat Bread",
             "Orange Juice", "Cheddar Cheese", "Almond Butter", "Brown Rice",
             "Sparkling Water", "Granola Cereal", "Olive Oil", "Tomato Sauce",
             "Frozen Pizza", "Protein Bar", "Green Tea", "Pasta Sauce",
             "Coconut Water", "Dark Chocolate", "Peanut Butter", "Apple Cider"]

    brands = ["NatureCo", "FreshFarm", "OrganicLife", "PureEats", "GreenValley",
              "HealthyChoice", None, "", "  freshfarm", "NATURECO", "organic life"]

    date_formats = [
        "2024-01-15", "Jan 15, 2024", "15/01/2024", "01-15-2024",
        "2024/01/15", "January 15 2024", "15 Jan 2024", None, "",
        "2024-13-45",  # Invalid date
    ]

    serving_formats = ["100g", "100 g", "100 grams", "3.5 oz", "0.1 kg",
                       "100ml", "1 cup", "2 tbsp", None, "", "about 100g", "100"]

    rows = []
    for i in range(n):
        name = random.choice(names) if random.random() > 0.05 else None
        # Occasionally duplicate exact rows
        if i > 10 and random.random() < 0.08:
            rows.append(rows[random.randint(0, len(rows)-1)].copy())
            continue

        rows.append({
            "product_name": name if random.random() > 0.03 else (name.upper() if name else None),
            "brands": random.choice(brands),
            "categories": random.choice(["Snacks", "Beverages", "Dairy", "snacks",
                                          "BEVERAGES", "dairy products", None, ""]),
            "quantity": random.choice(serving_formats),
            "serving_size": random.choice(serving_formats),
            "energy_100g": (random.uniform(10, 900) if random.random() > 0.15
                           else random.choice([None, "", "N/A", "-1", "unknown"])),
            "fat_100g": (round(random.uniform(0, 50), 1) if random.random() > 0.12
                         else random.choice([None, "", "trace", "-"])),
            "sugars_100g": (round(random.uniform(0, 60), 1) if random.random() > 0.1
                            else random.choice([None, "0,5", "12.3g", ""])),
            "proteins_100g": (round(random.uniform(0, 30), 1) if random.random() > 0.1
                              else random.choice([None, "", "high"])),
            "nutrition_grade_fr": random.choice(["a", "b", "c", "d", "e", "A", "B", None, "", "unknown"]),
            "countries": random.choice(["France", "france", "FR", "United States",
                                         "us", "USA", "Germany", None, ""]),
            "last_modified_datetime": random.choice(date_formats),
            "code": str(random.randint(1000000000000, 9999999999999)) if random.random() > 0.05 else None,
            "packaging": random.choice(["Plastic", "plastic", "PLASTIC", "Glass",
                                         "Cardboard", "cardboard box", None, ""]),
        })

    return pd.DataFrame(rows)


if df_raw is None:
    df_raw = generate_messy_data(200)

# Standardize — keep only the columns we care about
KEEP_COLUMNS = [
    "product_name", "brands", "categories", "quantity", "serving_size",
    "energy_100g", "fat_100g", "sugars_100g", "proteins_100g",
    "nutrition_grade_fr", "countries", "last_modified_datetime", "code", "packaging"
]
available_cols = [c for c in KEEP_COLUMNS if c in df_raw.columns]
df_raw = df_raw[available_cols].copy()

# Replace empty strings with NaN for consistency
df_raw = df_raw.replace({"": np.nan, " ": np.nan})

print(f"\nDataset: {len(df_raw)} rows x {len(df_raw.columns)} columns")
print(f"Source: {DATA_SOURCE}")

### Explore the mess

In [ ]:
# This is the "before" picture. Let's see exactly how bad it is.

print("=" * 60)
print("DATA QUALITY SNAPSHOT (Before Cleaning)")
print("=" * 60)

print(f"\nShape: {df_raw.shape}")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

print("\n-- Null counts per column --")
null_counts = df_raw.isnull().sum()
null_pcts = (df_raw.isnull().mean() * 100).round(1)
for col in df_raw.columns:
    bar = "█" * int(null_pcts[col] / 5)
    print(f"  {col:30s} {null_counts[col]:4d} nulls ({null_pcts[col]:5.1f}%) {bar}")

print(f"\nTotal null values: {df_raw.isnull().sum().sum()}")
print(f"Total cells: {df_raw.shape[0] * df_raw.shape[1]}")
print(f"Null rate: {df_raw.isnull().sum().sum() / (df_raw.shape[0] * df_raw.shape[1]) * 100:.1f}%")

### Show sample messy rows

In [ ]:
print("\n-- Sample rows (notice the inconsistencies) --\n")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 30)
print(df_raw.sample(10, random_state=42).to_string())

### Highlight specific mess types

In [ ]:
print("\n-- Examples of messy values --\n")

for col in ["serving_size", "energy_100g", "nutrition_grade_fr", "countries", "brands"]:
    if col in df_raw.columns:
        unique_vals = df_raw[col].dropna().unique()[:8]
        print(f"  {col}: {list(unique_vals)}")

### Save raw data

In [ ]:
raw_path = DATA_DIR / "raw_data.csv"
df_raw.to_csv(raw_path, index=False)

# Also save as JSON for the pipeline
meta = {
    "source": DATA_SOURCE,
    "rows": len(df_raw),
    "columns": list(df_raw.columns),
    "null_total": int(df_raw.isnull().sum().sum()),
    "duplicate_rows": int(df_raw.duplicated().sum()),
}
with open(DATA_DIR / "raw_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nSaved raw data to {raw_path}")
print(f"Saved metadata to {DATA_DIR / 'raw_meta.json'}")
print("\nReady for Notebook 02 (Agent Pipeline)")